We would like to predict the amount of electricity produced by a windfarm, as a function of the information gathered in a number of physical sensors (e.g. speed of the wind, temperature, ...).

The dataset is stored in project/data/regression/, similarly to 1.2.1 and the instructions are the same (compare at least two models and discuss the hyperparameter optimization procedure). Your objective is to obtain a R2 score superior to 0.85 on the test set, for at least 1 of your models.
The same remark about the test set, presented in exercice 1.2.1 also applies here.

Several methods might work, including some methods that we have nt explicitely studied in the class. Do not hesitate to try such methods.
Indication : a solution, with the correct hyperparameters, exists in scikit among the following scikit classes :
- linear_model.Ridge
- linear_model.Lasso
- neural_network.MLPRegressor
- svm.SVR, ensemble.AdaBoostRegressor

In [1]:
import numpy as np

X_train = np.load("regression/X_train.npy")
y_train = np.load("regression/y_train.npy")

X_test = np.load("regression/X_test.npy")
y_test = np.load("regression/y_test.npy")

X_train.shape, X_test.shape

((200, 200), (200, 200))

In [2]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

y_train = y_train.ravel()
y_test = y_test.ravel()

cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [3]:
from sklearn.linear_model import Ridge

ridge_search = GridSearchCV(
    estimator=Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge())
    ]),
    param_grid={
        "model__alpha": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
    },
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

ridge_search.fit(X_train, y_train)

ridge_search.best_params_, ridge_search.best_score_

({'model__alpha': 10.0}, np.float64(0.7216551792399422))

In [4]:
from sklearn.linear_model import Lasso

lasso_search = GridSearchCV(
    estimator=Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(max_iter=10000))
    ]),
    param_grid={
        "model__alpha": [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]
    },
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

lasso_search.fit(X_train, y_train)

lasso_search.best_params_, lasso_search.best_score_

({'model__alpha': 0.01}, np.float64(0.9283188308511925))

In [5]:
from sklearn.svm import SVR

svr_search = GridSearchCV(
    estimator=Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(kernel="rbf"))
    ]),
    param_grid={
        "model__C": [0.1, 1, 10, 100, 1000],
        "model__gamma": ["scale", 1, 0.1, 0.01, 0.001],
        "model__epsilon": [0.01, 0.1, 0.5]
    },
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

svr_search.fit(X_train, y_train)

svr_search.best_params_, svr_search.best_score_

({'model__C': 100, 'model__epsilon': 0.01, 'model__gamma': 0.001},
 np.float64(0.646193084571662))

In [6]:
from sklearn.neural_network import MLPRegressor

mlp_search = GridSearchCV(
    estimator=Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPRegressor(max_iter=3000, random_state=42, early_stopping=True))
    ]),
    param_grid={
        "model__hidden_layer_sizes": [(32,), (64,), (64, 32)],
        "model__alpha": [1e-4, 1e-3, 1e-2],
        "model__learning_rate_init": [1e-3, 1e-2]
    },
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

mlp_search.fit(X_train, y_train)

mlp_search.best_params_, mlp_search.best_score_

({'model__alpha': 0.01,
  'model__hidden_layer_sizes': (64, 32),
  'model__learning_rate_init': 0.01},
 np.float64(-0.9917584372174435))

In [7]:
model_scores = {
    "Ridge": ridge_search.best_score_,
    "Lasso": lasso_search.best_score_,
    "SVR": svr_search.best_score_,
    "MLP": mlp_search.best_score_
}

best_model_name = max(model_scores, key=model_scores.get)
best_search = {
    "Ridge": ridge_search,
    "Lasso": lasso_search,
    "SVR": svr_search,
    "MLP": mlp_search
}[best_model_name]

best_model_name, best_search.best_params_, model_scores

('Lasso',
 {'model__alpha': 0.01},
 {'Ridge': np.float64(0.7216551792399422),
  'Lasso': np.float64(0.9283188308511925),
  'SVR': np.float64(0.646193084571662),
  'MLP': np.float64(-0.9917584372174435)})

In [8]:
final_model = best_search.best_estimator_
final_model.fit(X_train, y_train)
final_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",0.01
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True


In [9]:
from sklearn.metrics import r2_score

test_preds = final_model.predict(X_test)
test_r2 = r2_score(y_test, test_preds)

test_r2

0.939614907187149

# Interpretation - Windfarm Power Production Regression

## Why the notebook was corrected

The previous conclusion was inconsistent with the notebook outputs: it claimed that nonlinear models outperformed linear ones and that the final test score exceeded 0.85, while the displayed scores did not support either statement.

The notebook now selects the final model from the computed validation scores instead of hardcoding an SVR, and it includes **Lasso** in the comparison because a linear model with the right regularization can be sufficient for this exercise.

## Hyperparameter justification

All models are evaluated inside a pipeline with `StandardScaler`. This is important because the sensor features can have very different scales, and regularized linear models, SVR and MLP are all sensitive to that.

### Ridge

Ridge is a linear baseline with L2 regularization. The grid explores several values of `alpha` over multiple orders of magnitude to control the regularization strength.

### Lasso

Lasso is also linear, but with L1 regularization. It can perform implicit feature selection by shrinking some coefficients to zero, which can be helpful when many sensor variables are irrelevant or redundant. The tested `alpha` values range from very weak to strong regularization.

### SVR

SVR with an RBF kernel is included to test a nonlinear alternative. The grid tunes `C`, `gamma` and `epsilon` because they directly control the flexibility of the function and the tolerance to residual errors.

### MLP

The neural network is treated as an additional nonlinear candidate, but it should only be considered better if its validation score is actually higher than the linear and kernel baselines. The conclusion must therefore follow the measured scores, not a general expectation about model complexity.

## Model selection

Each model is tuned with the same 5-fold cross-validation and compared with the same `R^2` metric. The selected final model is the one with the best mean cross-validated validation score.

This directly fixes the previous issue: the final estimator is no longer forced to be SVR if Ridge or Lasso performs better. If Lasso is the best model after tuning, then the notebook will now keep Lasso as the final model, which is consistent with the remark that the optimal method can be linear.

## Conclusion

The correct conclusion is model-dependent: after execution, the notebook should report which tuned model achieved the highest cross-validated `R^2`, then give the corresponding test `R^2` without overstating the result.

This fixes both remarks: the conclusion is now aligned with the computed scores, and the notebook no longer suggests that nonlinear models are automatically better when a tuned linear model may in fact be the best choice.
